# AST Laughter Labeler for 620 Videos

Uses **MIT/ast-finetuned-audioset-10-10-0** (1.2M downloads) to detect laughter in 10-second audio clips.

AudioSet includes 'Laughter' class! This gives us proper labels for our 620 videos.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

!pip install -q torch torchaudio librosa numpy pandas
!pip install -q transformers accelerate

import torch
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import os

BASE = '/content/drive/MyDrive/chuckle_net'
AUDIO_DIR = f'{BASE}/audio'
OUTPUT_DIR = f'{BASE}/ast_labels'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Audio dir: {AUDIO_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

# List audio files
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a')]
print(f'Found {len(audio_files)} audio files')

In [ ]:
# === LOAD AST MODEL ===
# Using Audio Spectrogram Transformer fine-tuned on AudioSet
# AudioSet has 527 classes including 'Laughter'

print('Loading AST model...')
model_name = 'MIT/ast-finetuned-audioset-10-10-0.4593'

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name)
model.eval()

# AudioSet class 261 = Laughter
# Check class labels
if hasattr(model.config, 'id2label'):
    print(f'Model has {len(model.config.id2label)} classes')
    # Find laughter class
    laughter_class = None
    for cid, label in model.config.id2label.items():
        if 'laugh' in label.lower():
            laughter_class = int(cid)
            print(f'Found laughter class: {cid} = {label}')
            break
    if laughter_class is None:
        print('WARNING: No laughter class found! Using class 261 as fallback')
        laughter_class = 261
else:
    laughter_class = 261
    print(f'Using default laughter class: 261')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Model loaded on {device}')

In [ ]:
# === EXTRACT 10-SEC CLIPS FROM VIDEO ===
def extract_clips_from_audio(audio_path, clip_duration=10.0, overlap=2.0):
    """Extract 10-second clips with 8s overlap from audio file."""
    try:
        # Load audio
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        duration = len(y) / sr
        
        clips = []
        for start in np.arange(0, duration - clip_duration, overlap):
            end = start + clip_duration
            clip_y = y[int(start * sr):int(end * sr)]
            clips.append({
                'start': start,
                'end': end,
                'waveform': clip_y,
                'duration': clip_duration
            })
        return clips
    except Exception as e:
        print(f'Error loading {audio_path}: {e}')
        return []

def predict_laughter_batch(clips, batch_size=8):
    """Predict laughter for a batch of clips."""
    predictions = []
    
    for i in range(0, len(clips), batch_size):
        batch = clips[i:i+batch_size]
        
        # Process each clip
        inputs = []
        for clip in batch:
            # Resample to 16000 if needed
            waveform = clip['waveform']
            if len(waveform) < 16000 * clip['duration']:
                waveform = np.pad(waveform, (0, int(16000 * clip['duration']) - len(waveform)))
            
            # Convert to array
            input_tensor = torch.FloatTensor(waveform[:int(16000 * clip['duration'])]).unsqueeze(0)
            inputs.append(input_tensor)
        
        # Stack into batch
        batch_tensor = torch.cat(inputs, dim=0)
        
        # Predict
        with torch.no_grad():
            outputs = model(batch_tensor.to(device))
            logits = outputs.logits  # (batch, num_classes)
            probs = torch.softmax(logits, dim=-1)
            laughter_probs = probs[:, laughter_class].cpu().numpy()
        
        for j, clip in enumerate(batch):
            clip['laughter_prob'] = float(laughter_probs[j])
            predictions.append(clip)
    
    return predictions

print('Functions defined')

In [ ]:
# === PROCESS ALL VIDEOS ===
all_results = []

for audio_file in tqdm(audio_files, desc='Processing videos'):
    video_id = audio_file.replace('.m4a', '')
    output_file = f'{OUTPUT_DIR}/{video_id}.json'
    
    # Skip if already processed
    if os.path.exists(output_file):
        print(f'Skipping {video_id} (already processed)')
        with open(output_file) as f:
            all_results.append(json.load(f))
        continue
    
    audio_path = f'{AUDIO_DIR}/{audio_file}'
    
    # Extract clips
    clips = extract_clips_from_audio(audio_path)
    
    if not clips:
        print(f'No clips for {video_id}')
        continue
    
    # Predict
    predictions = predict_laughter_batch(clips)
    
    # Save result
    result = {
        'video_id': video_id,
        'n_clips': len(predictions),
        'clips': [
            {'start': p['start'], 'end': p['end'], 'laughter_prob': p['laughter_prob']}
            for p in predictions
        ]
        # Summary stats
        'mean_laugh_prob': np.mean([p['laughter_prob'] for p in predictions]),
        'max_laugh_prob': np.max([p['laughter_prob'] for p in predictions]),
        'n_high_prob': sum(1 for p in predictions if p['laughter_prob'] > 0.1),
        'n_positive': sum(1 for p in predictions if p['laughter_prob'] > 0.1)
    }
    
    with open(output_file, 'w') as f:
        json.dump(result, f)
    
    all_results.append(result)

print(f'\nProcessed {len(all_results)} videos')

In [ ]:
# === ANALYZE RESULTS ===
import pandas as pd

df = pd.DataFrame(all_results)
print('=== LABEL STATISTICS ===')
print(f'Total videos: {len(df)}')
print(f'Mean laughter prob: {df["mean_laugh_prob"].mean():.4f}')
print(f'Max laughter prob: {df["max_laugh_prob"].max():.4f}')
print(f'Videos with high prob (>0.1): {(df["max_laugh_prob"] > 0.1).sum()}')
print(f'Videos with high prob (>0.5): {(df["max_laugh_prob"] > 0.5).sum()}')

# Overall clip-level stats
all_clips = []
for r in all_results:
    for clip in r['clips']:
        all_clips.append({
            'video_id': r['video_id'],
            'start': clip['start'],
            'end': clip['end'],
            'laughter_prob': clip['laughter_prob']
        })
clips_df = pd.DataFrame(all_clips)
print(f'\nTotal clips: {len(clips_df)}')
print(f'Clips with laughter (>0.1): {(clips_df["laughter_prob"] > 0.1).sum()}')
print(f'Positive rate: {(clips_df["laughter_prob"] > 0.1).mean():.1%}')

# Save summary
df.to_csv(f'{BASE}/ast_label_summary.csv', index=False)
print(f'\nSaved summary to {BASE}/ast_label_summary.csv')

In [ ]:
# === EXTRACT FEATURES FOR HIGH-PROB CLIPS ===
# For clips with laughter_prob > threshold, extract WavLM/prosody features
# Then train classifier

THRESHOLD = 0.1  # Label clip as positive if prob > 0.1

labeled_clips = []
for r in all_results:
    for clip in r['clips']:
        labeled_clips.append({
            'video_id': r['video_id'],
            'start': clip['start'],
            'end': clip['end'],
            'laughter_prob': clip['laughter_prob'],
            'label': 1 if clip['laughter_prob'] > THRESHOLD else 0
        })

labeled_df = pd.DataFrame(labeled_clips)
print(f'Total clips: {len(labeled_df)}')
print(f'Positive clips: {labeled_df["label"].sum()} ({100*labeled_df["label"].mean():.1f}%)')

# Save labeled clips
labeled_df.to_csv(f'{BASE}/ast_labeled_clips.csv', index=False)
print(f'Saved to {BASE}/ast_labeled_clips.csv')

In [ ]:
# === TRAIN CLASSIFIER ON AST LABELS ===
# Extract prosody features + train classifier

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score

def extract_prosody(waveform, sr=22050):
    """Extract 23-dim prosody features from waveform."""
    features = []
    
    # F0 (pitch) using librosa
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            waveform, fmin=50, fmax=500, sr=sr
        )
        f0_mean = np.nanmean(f0) if not np.all(np.isnan(f0)) else 0
        f0_std = np.nanstd(f0) if not np.all(np.isnan(f0)) else 0
        voiced_rate = np.mean(voiced_flag) if len(voiced_flag) > 0 else 0
        voiced_probs_mean = np.nanmean(voiced_probs) if len(voiced_probs) > 0 else 0
    except:
        f0_mean = f0_std = voiced_rate = voiced_probs_mean = 0
    
    features.extend([f0_mean, f0_std, voiced_rate, voiced_probs_mean])
    
    # Energy
    rms = librosa.feature.rms(y=waveform)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms)])
    
    # MFCCs
    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13)
    for i in range(13):
        features.extend([np.mean(mfccs[i]), np.std(mfccs[i])])
    
    # Spectral features
    spec_cent = librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=waveform, sr=sr)[0]
    features.extend([np.mean(spec_cent), np.std(spec_cent)])
    features.extend([np.mean(spec_bw), np.std(spec_bw)])
    
    return np.array(features, dtype=np.float32)

# Extract features for a subset (for speed)
print('Extracting features for training...')
X_list = []
y_list = []
video_ids = []

# Process in batches
for idx, row in tqdm(labeled_df.iterrows(), total=len(labeled_df), desc='Extracting'):
    video_id = row['video_id']
    start, end = row['start'], row['end']
    label = row['label']
    
    audio_path = f'{AUDIO_DIR}/{video_id}.m4a'
    if not os.path.exists(audio_path):
        continue
    
    try:
        # Load clip
        y, sr = librosa.load(audio_path, offset=start, duration=end-start, sr=22050, mono=True)
        
        # Extract features
        feat = extract_prosody(y, sr)
        
        X_list.append(feat)
        y_list.append(label)
        video_ids.append(video_id)
    except Exception as e:
        continue

X = np.array(X_list)
y = np.array(y_list)
print(f'Extracted {len(X)} samples, pos={y.sum()} ({100*y.mean():.1f}%)')

In [ ]:
# === TRAIN AND EVALUATE ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split by video for fair evaluation
unique_videos = list(set(video_ids))
train_vids, test_vids = train_test_split(unique_videos, test_size=0.2, random_state=42)

train_mask = [v in train_vids for v in video_ids]
test_mask = [v in test_vids for v in video_ids]

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train: {len(X_train)} samples, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Test: {len(X_test)} samples, pos={y_test.sum()} ({100*y_test.mean():.1f}%)')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

# Evaluate
y_pred = clf.predict(X_test_scaled)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== TEST RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')

# Save model
import pickle
with open(f'{BASE}/ast_prosody_model.pkl', 'wb') as f:
    pickle.dump({'model': clf, 'scaler': scaler}, f)
print(f'\nModel saved to {BASE}/ast_prosody_model.pkl')